# Creating Bronze Delta tables

In [0]:
# File location and type
file_location = "/Volumes/workspace/default/delta_tables_training/StockItem.csv"
file_type = 'csv'

# CSV options
infer_schema = 'true'
first_row_is_header = 'true'
delimiter = ',' 

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("delimiter", delimiter) \
  .load(file_location)

display(df)

In [0]:
# Create a view or table
temp_table_name = 'StockItem_csv'
df.createOrReplaceTempView(temp_table_name)

In [0]:
%sql
select * from StockItem_csv

In [0]:
# Creating an additional column to store the update date

from pyspark.sql.functions import current_timestamp
df2 = df.withColumn("update_date", current_timestamp())

display(df2)

In [0]:
# Creating a delta table
spark.sql("create database if not exists bronze_wwi")
spark.sql("use bronze_wwi")
spark.sql("drop table if exists stockitems")
df2.write.format("delta").mode("append").saveAsTable("stockitems")

In [0]:
# Defining static schema

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

schema = StructType([
  StructField("StockItemID", IntegerType(), True),
  StructField("StockItemName", StringType(), True),
  StructField("SupplierID", IntegerType(), True),
  StructField("ColorID", IntegerType(), True),
  StructField("UnitPackageID", IntegerType(), True),
  StructField("OuterPackageID", IntegerType(), True),
  StructField("Brand", StringType(), True),
  StructField("Size", StringType(), True),
  StructField("LeadTimeDays", IntegerType(), True),
  StructField("QuantityPerOuter", IntegerType(), True),
  StructField("Barcode", StringType(), True),
  StructField("TaxRate", DoubleType(), True),
  StructField("UnitPrice", DoubleType(), True),
  StructField("RecommendedRetailPrice", DoubleType(), True),
  StructField("TypicalWeightPerUnit", DoubleType(), True),
  StructField("ValidFrom", DateType(), True),
  StructField("ValidTo", DateType(), True),
  StructField("update_date", DateType(), True)
])

display(schema)

In [0]:
# Creating the bronze delta table in Overwrite mode
file_location = "/Volumes/workspace/default/delta_tables_training/StockItem.csv"

df = spark.read.csv(file_location, schema=schema, header=True)

df_new = df.withColumn("update_date", current_timestamp())

spark.sql("create database if not exists bronze_wwi")
spark.sql("use bronze_wwi")
spark.sql("drop table if exists stockitems")
df_new.write.format("delta").mode("overwrite").saveAsTable("stockitems")

In [0]:
display(df_new)